# SatQuery AI — Remote-Sensing VLM Bake-Off on Google Colab

**Candidates Evaluated:**
1. `AdaptLLM/remote-sensing-Qwen2-VL-2B-Instruct` (Domain-Adapted Remote Sensing Candidate)
2. `Qwen/Qwen2-VL-2B-Instruct` (Generalist Multimodal Baseline)

> **Evaluation Constraints:**
> - 40 Deterministic Remote-Sensing Questions (20 categories x 2 questions)
> - Sequential execution: Strict single-model memory residency (one model in VRAM at a time)
> - Two-stage protocol: Mandatory 1-question smoke test before 40-question batch
> - Results exported to `satquery_rsvlm_colab_results.json`

In [ ]:
# 1. Environment Setup & Dependency Installation
!pip install -q torch torchvision "transformers>=4.45.0" accelerate qwen-vl-utils pillow

import torch
assert torch.cuda.is_available(), 'Please select a GPU runtime (Runtime -> Change runtime type -> T4 GPU)'
print(f'Active GPU: {torch.cuda.get_device_name(0)}')
print(f'Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')

In [ ]:
# 2. Workspace & Imagery Setup
# Places and verifies all 5 required evaluation images under backend/data/ before Stage 1:
# - backend/data/default_aerial.jpg
# - backend/data/levir_cd/A/test_2_0000_0000.png
# - backend/data/levir_cd/B/test_2_0000_0000.png
# - backend/data/optical_sar/optical.tif
# - backend/data/optical_sar/sar.tif

import base64
import io
import os
import tarfile

REQUIRED_IMAGES = [
    'backend/data/default_aerial.jpg',
    'backend/data/levir_cd/A/test_2_0000_0000.png',
    'backend/data/levir_cd/B/test_2_0000_0000.png',
    'backend/data/optical_sar/optical.tif',
    'backend/data/optical_sar/sar.tif',
]

def check_all_images_valid():
    for p in REQUIRED_IMAGES:
        if not os.path.exists(p) or os.path.getsize(p) < 1000:
            return False
    return True

# Embedded verified imagery payload (genuine evaluation imagery bundle)
IMAGERY_ARCHIVE_B64 = "..."

if not check_all_images_valid():
    print('Unpacking verified evaluation imagery archive into backend/data/...')
    archive_bytes = base64.b64decode(IMAGERY_ARCHIVE_B64)
    with tarfile.open(fileobj=io.BytesIO(archive_bytes), mode='r:gz') as tar:
        tar.extractall('.')
        try:
            tar.extractall('/content')
        except Exception:
            pass
    print('Evaluation imagery unpacked successfully.')
else:
    print('Evaluation imagery already present and verified.')
    import shutil
    for p in REQUIRED_IMAGES:
        target_p = os.path.join('/content', p)
        if os.path.exists(p) and not os.path.exists(target_p):
            try:
                os.makedirs(os.path.dirname(target_p), exist_ok=True)
                shutil.copy2(p, target_p)
            except Exception:
                pass

# Verify all 5 files
print('\nEvaluation imagery status:')
all_ok = True
for p in REQUIRED_IMAGES:
    if os.path.exists(p):
        sz = os.path.getsize(p)
        status = 'OK' if sz > 1000 else 'INVALID_POINTER'
        if sz <= 1000:
            all_ok = False
        print(f'  [{status}] {p} ({sz:,} bytes)')
    else:
        all_ok = False
        print(f'  [MISSING] {p}')

assert all_ok, 'Evaluation imagery setup failed: one or more required images are missing or invalid.'
print('\nAll 5 evaluation images verified and ready for Colab runner.')


In [ ]:
# 3. Load Embedded 40-Question Evaluation Suite
import json
QUESTIONS_JSON_STR = "[\n  {\n    \"question_id\": \"RS-SCN-01\",\n    \"category\": \"scene_description\",\n    \"category_name\": \"Scene Description\",\n    \"question_text\": \"Describe the overall scene and dominant landscape characteristics visible in this satellite image.\",\n    \"input_type\": \"single\",\n    \"image_role\": \"optical\",\n    \"sample_id\": \"sample_optical_aerial\",\n    \"ground_truth_available\": false,\n    \"ground_truth_answer\": null\n  },\n  {\n    \"question_id\": \"RS-SCN-02\",\n    \"category\": \"scene_description\",\n    \"category_name\": \"Scene Description\",\n    \"question_text\": \"What are the primary natural and man-made elements visible across this scene?\",\n    \"input_type\": \"single\",\n    \"image_role\": \"optical\",\n    \"sample_id\": \"sample_optical_aerial\",\n    \"ground_truth_available\": false,\n    \"ground_truth_answer\": null\n  },\n  {\n    \"question_id\": \"RS-LND-01\",\n    \"category\": \"land_cover\",\n    \"category_name\": \"Land Cover\",\n    \"question_text\": \"What types of land cover are visible in this satellite image?\",\n    \"input_type\": \"single\",\n    \"image_role\": \"optical\",\n    \"sample_id\": \"sample_optical_aerial\",\n    \"ground_truth_available\": false,\n    \"ground_truth_answer\": null\n  },\n  {\n    \"question_id\": \"RS-LND-02\",\n    \"category\": \"land_cover\",\n    \"category_name\": \"Land Cover\",\n    \"question_text\": \"What is the dominant land-use or land-cover class across the observed area?\",\n    \"input_type\": \"single\",\n    \"image_role\": \"optical\",\n    \"sample_id\": \"sample_optical_aerial\",\n    \"ground_truth_available\": false,\n    \"ground_truth_answer\": null\n  },\n  {\n    \"question_id\": \"RS-VEG-01\",\n    \"category\": \"vegetation\",\n    \"category_name\": \"Vegetation\",\n    \"question_text\": \"Is vegetation present in this image, and if so, what type of canopy or ground vegetation is discernible?\",\n    \"input_type\": \"single\",\n    \"image_role\": \"optical\",\n    \"sample_id\": \"sample_optical_aerial\",\n    \"ground_truth_available\": false,\n    \"ground_truth_answer\": null\n  },\n  {\n    \"question_id\": \"RS-VEG-02\",\n    \"category\": \"vegetation\",\n    \"category_name\": \"Vegetation\",\n    \"question_text\": \"Are there visible tree canopies, green spaces, or vegetated boundaries within the scene?\",\n    \"input_type\": \"single\",\n    \"image_role\": \"optical\",\n    \"sample_id\": \"sample_optical_aerial\",\n    \"ground_truth_available\": false,\n    \"ground_truth_answer\": null\n  },\n  {\n    \"question_id\": \"RS-WTR-01\",\n    \"category\": \"water\",\n    \"category_name\": \"Water\",\n    \"question_text\": \"Is there any visible surface water, such as a river, canal, lake, or retention pond in this image?\",\n    \"input_type\": \"single\",\n    \"image_role\": \"optical\",\n    \"sample_id\": \"sample_optical_aerial\",\n    \"ground_truth_available\": false,\n    \"ground_truth_answer\": null\n  },\n  {\n    \"question_id\": \"RS-WTR-02\",\n    \"category\": \"water\",\n    \"category_name\": \"Water\",\n    \"question_text\": \"Are drainage channels, water bodies, or wetlands observable in this satellite view?\",\n    \"input_type\": \"single\",\n    \"image_role\": \"optical\",\n    \"sample_id\": \"sample_optical_aerial\",\n    \"ground_truth_available\": false,\n    \"ground_truth_answer\": null\n  },\n  {\n    \"question_id\": \"RS-AGR-01\",\n    \"category\": \"agriculture\",\n    \"category_name\": \"Agriculture\",\n    \"question_text\": \"Does this scene appear agricultural, or are cultivated crop parcels and farming plots visible?\",\n    \"input_type\": \"single\",\n    \"image_role\": \"optical\",\n    \"sample_id\": \"sample_optical_aerial\",\n    \"ground_truth_available\": false,\n    \"ground_truth_answer\": null\n  },\n  {\n    \"question_id\": \"RS-AGR-02\",\n    \"category\": \"agriculture\",\n    \"category_name\": \"Agriculture\",\n    \"question_text\": \"Are there regular geometric field boundaries, furrow patterns, or agricultural irrigation signatures?\",\n    \"input_type\": \"single\",\n    \"image_role\": \"optical\",\n    \"sample_id\": \"sample_optical_aerial\",\n    \"ground_truth_available\": false,\n    \"ground_truth_answer\": null\n  },\n  {\n    \"question_id\": \"RS-BLD-01\",\n    \"category\": \"buildings\",\n    \"category_name\": \"Buildings\",\n    \"question_text\": \"Are buildings or residential structures visible in this scene?\",\n    \"input_type\": \"single\",\n    \"image_role\": \"optical\",\n    \"sample_id\": \"sample_optical_aerial\",\n    \"ground_truth_available\": false,\n    \"ground_truth_answer\": null\n  },\n  {\n    \"question_id\": \"RS-BLD-02\",\n    \"category\": \"buildings\",\n    \"category_name\": \"Buildings\",\n    \"question_text\": \"What can you deduce about the density, layout, or roof structures of the buildings in this image?\",\n    \"input_type\": \"single\",\n    \"image_role\": \"optical\",\n    \"sample_id\": \"sample_optical_aerial\",\n    \"ground_truth_available\": false,\n    \"ground_truth_answer\": null\n  },\n  {\n    \"question_id\": \"RS-INF-01\",\n    \"category\": \"infrastructure\",\n    \"category_name\": \"Infrastructure\",\n    \"question_text\": \"What civil, transportation, or utility infrastructure is visible in this scene?\",\n    \"input_type\": \"single\",\n    \"image_role\": \"optical\",\n    \"sample_id\": \"sample_optical_aerial\",\n    \"ground_truth_available\": false,\n    \"ground_truth_answer\": null\n  },\n  {\n    \"question_id\": \"RS-INF-02\",\n    \"category\": \"infrastructure\",\n    \"category_name\": \"Infrastructure\",\n    \"question_text\": \"Are engineered structures, industrial compounds, or utility corridors present?\",\n    \"input_type\": \"single\",\n    \"image_role\": \"optical\",\n    \"sample_id\": \"sample_optical_aerial\",\n    \"ground_truth_available\": false,\n    \"ground_truth_answer\": null\n  },\n  {\n    \"question_id\": \"RS-ROD-01\",\n    \"category\": \"roads\",\n    \"category_name\": \"Roads\",\n    \"question_text\": \"Are roads, paved streets, or transport networks visible in this image?\",\n    \"input_type\": \"single\",\n    \"image_role\": \"optical\",\n    \"sample_id\": \"sample_optical_aerial\",\n    \"ground_truth_available\": false,\n    \"ground_truth_answer\": null\n  },\n  {\n    \"question_id\": \"RS-ROD-02\",\n    \"category\": \"roads\",\n    \"category_name\": \"Roads\",\n    \"question_text\": \"Describe the roadway network pattern, intersections, or pathways visible in the scene.\",\n    \"input_type\": \"single\",\n    \"image_role\": \"optical\",\n    \"sample_id\": \"sample_optical_aerial\",\n    \"ground_truth_available\": false,\n    \"ground_truth_answer\": null\n  },\n  {\n    \"question_id\": \"RS-SHP-01\",\n    \"category\": \"ships\",\n    \"category_name\": \"Ships\",\n    \"question_text\": \"Are there any ships, maritime vessels, or boats visible in this scene?\",\n    \"input_type\": \"single\",\n    \"image_role\": \"optical\",\n    \"sample_id\": \"sample_optical_aerial\",\n    \"ground_truth_available\": false,\n    \"ground_truth_answer\": null\n  },\n  {\n    \"question_id\": \"RS-SHP-02\",\n    \"category\": \"ships\",\n    \"category_name\": \"Ships\",\n    \"question_text\": \"Is there any harbor, dockside mooring, or marine traffic observable in the image?\",\n    \"input_type\": \"single\",\n    \"image_role\": \"optical\",\n    \"sample_id\": \"sample_optical_aerial\",\n    \"ground_truth_available\": false,\n    \"ground_truth_answer\": null\n  },\n  {\n    \"question_id\": \"RS-AIR-01\",\n    \"category\": \"aircraft\",\n    \"category_name\": \"Aircraft\",\n    \"question_text\": \"Are aircraft visible on the ground or in flight in this satellite image?\",\n    \"input_type\": \"single\",\n    \"image_role\": \"optical\",\n    \"sample_id\": \"sample_optical_aerial\",\n    \"ground_truth_available\": false,\n    \"ground_truth_answer\": null\n  },\n  {\n    \"question_id\": \"RS-AIR-02\",\n    \"category\": \"aircraft\",\n    \"category_name\": \"Aircraft\",\n    \"question_text\": \"Are runways, taxiways, hangars, or airport aprons observable?\",\n    \"input_type\": \"single\",\n    \"image_role\": \"optical\",\n    \"sample_id\": \"sample_optical_aerial\",\n    \"ground_truth_available\": false,\n    \"ground_truth_answer\": null\n  },\n  {\n    \"question_id\": \"RS-VEH-01\",\n    \"category\": \"vehicles\",\n    \"category_name\": \"Vehicles\",\n    \"question_text\": \"Are vehicles (such as cars, trucks, or heavy machinery) visible on roads or parking lots?\",\n    \"input_type\": \"single\",\n    \"image_role\": \"optical\",\n    \"sample_id\": \"sample_optical_aerial\",\n    \"ground_truth_available\": false,\n    \"ground_truth_answer\": null\n  },\n  {\n    \"question_id\": \"RS-VEH-02\",\n    \"category\": \"vehicles\",\n    \"category_name\": \"Vehicles\",\n    \"question_text\": \"Can individual ground transport vehicles or parking clusters be resolved at this image resolution?\",\n    \"input_type\": \"single\",\n    \"image_role\": \"optical\",\n    \"sample_id\": \"sample_optical_aerial\",\n    \"ground_truth_available\": false,\n    \"ground_truth_answer\": null\n  },\n  {\n    \"question_id\": \"RS-LOC-01\",\n    \"category\": \"spatial_location\",\n    \"category_name\": \"Spatial Location\",\n    \"question_text\": \"Where in the image (e.g., upper left, center, bottom right) are the most prominent structures located?\",\n    \"input_type\": \"single\",\n    \"image_role\": \"optical\",\n    \"sample_id\": \"sample_optical_aerial\",\n    \"ground_truth_available\": false,\n    \"ground_truth_answer\": null\n  },\n  {\n    \"question_id\": \"RS-LOC-02\",\n    \"category\": \"spatial_location\",\n    \"category_name\": \"Spatial Location\",\n    \"question_text\": \"In which quadrants or sectors of the image is vegetation concentrated?\",\n    \"input_type\": \"single\",\n    \"image_role\": \"optical\",\n    \"sample_id\": \"sample_optical_aerial\",\n    \"ground_truth_available\": false,\n    \"ground_truth_answer\": null\n  },\n  {\n    \"question_id\": \"RS-REL-01\",\n    \"category\": \"spatial_relationships\",\n    \"category_name\": \"Spatial Relationships\",\n    \"question_text\": \"Where are the buildings located relative to the roadways or adjacent open spaces?\",\n    \"input_type\": \"single\",\n    \"image_role\": \"optical\",\n    \"sample_id\": \"sample_optical_aerial\",\n    \"ground_truth_available\": false,\n    \"ground_truth_answer\": null\n  },\n  {\n    \"question_id\": \"RS-REL-02\",\n    \"category\": \"spatial_relationships\",\n    \"category_name\": \"Spatial Relationships\",\n    \"question_text\": \"Describe the spatial arrangement between built structures and surrounding natural ground cover.\",\n    \"input_type\": \"single\",\n    \"image_role\": \"optical\",\n    \"sample_id\": \"sample_optical_aerial\",\n    \"ground_truth_available\": false,\n    \"ground_truth_answer\": null\n  },\n  {\n    \"question_id\": \"RS-URB-01\",\n    \"category\": \"urban_scene_reasoning\",\n    \"category_name\": \"Urban Scene Reasoning\",\n    \"question_text\": \"Does this scene depict an urban, suburban, or planned human settlement?\",\n    \"input_type\": \"single\",\n    \"image_role\": \"optical\",\n    \"sample_id\": \"sample_optical_aerial\",\n    \"ground_truth_available\": false,\n    \"ground_truth_answer\": null\n  },\n  {\n    \"question_id\": \"RS-URB-02\",\n    \"category\": \"urban_scene_reasoning\",\n    \"category_name\": \"Urban Scene Reasoning\",\n    \"question_text\": \"What indicators of human activity, zoning, or urban density can be inferred?\",\n    \"input_type\": \"single\",\n    \"image_role\": \"optical\",\n    \"sample_id\": \"sample_optical_aerial\",\n    \"ground_truth_available\": false,\n    \"ground_truth_answer\": null\n  },\n  {\n    \"question_id\": \"RS-RUR-01\",\n    \"category\": \"rural_scene_reasoning\",\n    \"category_name\": \"Rural Scene Reasoning\",\n    \"question_text\": \"Does this landscape exhibit rural, open-country, or undeveloped terrain characteristics?\",\n    \"input_type\": \"single\",\n    \"image_role\": \"optical\",\n    \"sample_id\": \"sample_optical_aerial\",\n    \"ground_truth_available\": false,\n    \"ground_truth_answer\": null\n  },\n  {\n    \"question_id\": \"RS-RUR-02\",\n    \"category\": \"rural_scene_reasoning\",\n    \"category_name\": \"Rural Scene Reasoning\",\n    \"question_text\": \"What natural topographic features, unpaved surfaces, or rural plot divisions are visible?\",\n    \"input_type\": \"single\",\n    \"image_role\": \"optical\",\n    \"sample_id\": \"sample_optical_aerial\",\n    \"ground_truth_available\": false,\n    \"ground_truth_answer\": null\n  },\n  {\n    \"question_id\": \"RS-CAP-01\",\n    \"category\": \"captioning\",\n    \"category_name\": \"Captioning\",\n    \"question_text\": \"Generate a concise, factual remote-sensing caption summarizing this overhead image.\",\n    \"input_type\": \"single\",\n    \"image_role\": \"optical\",\n    \"sample_id\": \"sample_optical_aerial\",\n    \"ground_truth_available\": false,\n    \"ground_truth_answer\": null\n  },\n  {\n    \"question_id\": \"RS-CAP-02\",\n    \"category\": \"captioning\",\n    \"category_name\": \"Captioning\",\n    \"question_text\": \"Provide a formal satellite image descriptor documenting terrain, structures, and land use.\",\n    \"input_type\": \"single\",\n    \"image_role\": \"optical\",\n    \"sample_id\": \"sample_optical_aerial\",\n    \"ground_truth_available\": false,\n    \"ground_truth_answer\": null\n  },\n  {\n    \"question_id\": \"RS-GND-01\",\n    \"category\": \"grounding\",\n    \"category_name\": \"Visual Grounding\",\n    \"question_text\": \"Identify and specify the image regions or bounding coordinates where prominent buildings reside.\",\n    \"input_type\": \"single\",\n    \"image_role\": \"optical\",\n    \"sample_id\": \"sample_optical_aerial\",\n    \"ground_truth_available\": false,\n    \"ground_truth_answer\": null\n  },\n  {\n    \"question_id\": \"RS-GND-02\",\n    \"category\": \"grounding\",\n    \"category_name\": \"Visual Grounding\",\n    \"question_text\": \"Locate the principal roadway or access path across the scene by its spatial coordinates or quadrant.\",\n    \"input_type\": \"single\",\n    \"image_role\": \"optical\",\n    \"sample_id\": \"sample_optical_aerial\",\n    \"ground_truth_available\": false,\n    \"ground_truth_answer\": null\n  },\n  {\n    \"question_id\": \"RS-SAR-01\",\n    \"category\": \"sar_understanding\",\n    \"category_name\": \"SAR Understanding\",\n    \"question_text\": \"In this radar/SAR image, what features exhibit high microwave backscatter (bright return) versus low backscatter (dark return)?\",\n    \"input_type\": \"single\",\n    \"image_role\": \"sar\",\n    \"sample_id\": \"sample_sar_single\",\n    \"ground_truth_available\": false,\n    \"ground_truth_answer\": null\n  },\n  {\n    \"question_id\": \"RS-SAR-02\",\n    \"category\": \"sar_understanding\",\n    \"category_name\": \"SAR Understanding\",\n    \"question_text\": \"Comparing this optical and SAR pair, what complementary structural details or surface roughness characteristics are highlighted in radar?\",\n    \"input_type\": \"pair\",\n    \"image_role\": \"optical_sar\",\n    \"sample_id\": \"sample_optical_sar_pair\",\n    \"ground_truth_available\": false,\n    \"ground_truth_answer\": null\n  },\n  {\n    \"question_id\": \"RS-CHG-01\",\n    \"category\": \"temporal_change_reasoning\",\n    \"category_name\": \"Temporal Change Reasoning\",\n    \"question_text\": \"What significant architectural, structural, or surface changes appear between Image A (T1) and Image B (T2)?\",\n    \"input_type\": \"pair\",\n    \"image_role\": \"temporal_pair\",\n    \"sample_id\": \"sample_levir_cd_pair\",\n    \"ground_truth_available\": false,\n    \"ground_truth_answer\": null\n  },\n  {\n    \"question_id\": \"RS-CHG-02\",\n    \"category\": \"temporal_change_reasoning\",\n    \"category_name\": \"Temporal Change Reasoning\",\n    \"question_text\": \"Are there new buildings constructed or existing buildings removed between these two temporal acquisitions?\",\n    \"input_type\": \"pair\",\n    \"image_role\": \"temporal_pair\",\n    \"sample_id\": \"sample_levir_cd_pair\",\n    \"ground_truth_available\": false,\n    \"ground_truth_answer\": null\n  },\n  {\n    \"question_id\": \"RS-OPN-01\",\n    \"category\": \"open_vocabulary_rs\",\n    \"category_name\": \"Open-Vocabulary RS Reasoning\",\n    \"question_text\": \"What broader environmental, industrial, or developmental inferences can be drawn from this satellite view?\",\n    \"input_type\": \"single\",\n    \"image_role\": \"optical\",\n    \"sample_id\": \"sample_optical_aerial\",\n    \"ground_truth_available\": false,\n    \"ground_truth_answer\": null\n  },\n  {\n    \"question_id\": \"RS-OPN-02\",\n    \"category\": \"open_vocabulary_rs\",\n    \"category_name\": \"Open-Vocabulary RS Reasoning\",\n    \"question_text\": \"What potential environmental or operational risks (such as flood exposure or encroachment) can be hypothesized from this scene?\",\n    \"input_type\": \"single\",\n    \"image_role\": \"optical\",\n    \"sample_id\": \"sample_optical_aerial\",\n    \"ground_truth_available\": false,\n    \"ground_truth_answer\": null\n  }\n]"
QUESTIONS_DATA = json.loads(QUESTIONS_JSON_STR)
with open('questions_40.json', 'w', encoding='utf-8') as f:
    json.dump(QUESTIONS_DATA, f, indent=2)
print(f'Saved {len(QUESTIONS_DATA)} evaluation questions across 20 categories.')

In [ ]:
# 4. Write Standalone Evaluation Script
# The runner is base64-encoded here so the notebook JSON stays clean
# regardless of any quote characters inside the script itself.
import base64 as _b64, os as _os

_RUNNER_B64 = "..."
RUNNER_SCRIPT = _b64.b64decode(_RUNNER_B64).decode('utf-8')

target = '/content/colab_runner.py' if _os.path.exists('/content') else 'colab_runner.py'
with open(target, 'w', encoding='utf-8') as f:
    f.write(RUNNER_SCRIPT)
print(f'colab_runner.py written to {target} successfully.')

# Verify the OOM fix is present in the written file
assert '_SMOKE_MAX_PIXELS' in RUNNER_SCRIPT, 'OOM fix missing: _SMOKE_MAX_PIXELS'
assert 'max_pixels=_SMOKE_MAX_PIXELS' in RUNNER_SCRIPT, 'OOM fix missing: max_pixels='
assert 'torch.inference_mode()' in RUNNER_SCRIPT, 'OOM fix missing: inference_mode'
print('OOM fix verified in written colab_runner.py.')


In [ ]:
# 5. Verify colab_runner.py & Evaluation Imagery Before Running Evaluation
# This cell confirms the runner is syntactically valid and importable,
# and confirms all 5 evaluation image samples can be resolved into memory.
# It does NOT download model weights or run any model inference.
import os
import sys
import py_compile
import importlib.util

runner_path = '/content/colab_runner.py' if os.path.exists('/content/colab_runner.py') else 'colab_runner.py'
py_compile.compile(runner_path, doraise=True)
print('colab_runner.py syntax OK')

spec = importlib.util.spec_from_file_location('colab_runner', runner_path)
print(f'colab_runner.py importable: {spec.origin}')

q_file = '/content/questions_40.json' if os.path.exists('/content/questions_40.json') else 'questions_40.json'
assert os.path.exists(q_file), f'{q_file} does not exist'
print(f'{q_file} exists: True')

# Pre-flight check: Verify all 5 evaluation image samples can be loaded
sys.path.insert(0, os.path.dirname(os.path.abspath(runner_path)))
import colab_runner
samples = colab_runner.resolve_image_samples('backend/data')
assert samples['sample_optical_aerial']['image'] is not None, 'Optical aerial sample image failed to load.'
assert samples['sample_levir_cd_pair']['image_a'] is not None, 'LEVIR-CD pair image A failed to load.'
assert samples['sample_levir_cd_pair']['image_b'] is not None, 'LEVIR-CD pair image B failed to load.'
assert samples['sample_sar_single']['image'] is not None, 'SAR sample image failed to load.'
assert samples['sample_optical_sar_pair']['image_a'] is not None, 'Optical-SAR optical image failed to load.'
assert samples['sample_optical_sar_pair']['image_b'] is not None, 'Optical-SAR SAR image failed to load.'
print('Pre-flight imagery check PASSED. All 5 images successfully loaded.')
print('Pre-flight checks PASSED. Ready to run evaluation.')


In [ ]:
# 6. Run Full Evaluation Protocol
# Stage 1 (Smoke Tests) -> Stage 2 (40 Questions x 2 Models) -> Stage 3 (JSON Export) -> Stage 4 (Summary Table)
!python colab_runner.py backend/data questions_40.json satquery_rsvlm_colab_results.json

In [ ]:
# 7. Download Results File
from google.colab import files
files.download('satquery_rsvlm_colab_results.json')
print('satquery_rsvlm_colab_results.json ready for download.')